## Image Augmemntation:


**This is a list of Python modules that have been imported into the current Python session using the import statement. Each module provides additional functionality that can be used within a Python program.**

Here's a brief overview of each of the modules that have been imported:

- `os`: Provides a way to interact with the operating system, including working with file paths, directories, and environment variables.

- `uuid`: Provides functions for generating unique identifiers (UUIDs) based on the current time, a random number, or a combination of the two.

- `shutil`: Provides high-level file operations, such as copying, moving, and deleting files and directories.

- `numpy`: A Python library for working with multi-dimensional arrays and matrices. It also provides many mathematical functions for working with these data structures.

- `pandas`: A Python library for data manipulation and analysis. It provides data structures for efficiently storing and querying large datasets, as well as functions for cleaning, transforming, and merging data.

- `PIL.Image`: A module from the Pillow library that provides image processing functionality, such as opening, manipulating, and saving image files.

- `pathlib.Path`: A module for working with file paths. It provides a way to create and manipulate file paths in a platform-independent way.

- `random`: A module for generating random numbers and making random selections from lists.

- `PIL.ImageOps`: A module from the Pillow library that provides a set of image processing operations, such as flipping, rotating, and inverting images.

- `PIL.ImageEnhance`: A module from the Pillow library that provides a set of image enhancement operations, such as adjusting brightness, contrast, and sharpness.

- `PIL.ImageFilter`: A module from the Pillow library that provides a set of image filtering operations, such as blurring, sharpening, and edge detection.

- `tensorflow`: A Python library for machine learning and deep learning. It provides tools for building and training machine learning models, as well as tools for working with image and text data.

- `matplotlib.pyplot`: A module for creating data visualizations, such as line plots, scatter plots, and histograms.

- `imgaug.augmenters`: A module for performing data augmentation on images. It provides a set of functions for applying a wide range of image transformations, such as flipping, rotating, and zooming.

- `tensorflow.keras.preprocessing.image`: A module from the TensorFlow library that provides tools for preprocessing and augmenting images. It includes functions for loading images, converting them to arrays, and applying transformations such as rotation and scaling.






In [1]:
import os
import uuid
import shutil
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
import random
from PIL import ImageOps, ImageEnhance, ImageFilter
import tensorflow as tf
from matplotlib import pyplot as plt
import imgaug.augmenters as iaa
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

In [2]:
# Set seed values for random number generator and file system

seed_value = 42
os.environ['PYTHONHASHSEED'] = str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

In [3]:
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

**The code creates a copy of the directory HairSkin-Removed-File and all of its contents to a new directory called Augmented Image Forder using the `shutil.copytree()` function.
The `shutil.copytree()` function recursively copies an entire directory tree from the `src_dir` to the `dst_dir` while preserving the structure of the original directory. If the dst_dir already exists, it will raise a FileExistsError.**

In [5]:
src_dir = "HairSkin-Removed-File"
dst_dir = "Augmented Image Folder"


# Copy the entire directory to the destination directory
shutil.copytree(src_dir, dst_dir)

'Augmented Image Folder'

In [6]:
def count_images_in_subdirs(parent_dir):
    total_count = 0
    for sub_dir in os.listdir(parent_dir):
        sub_dir_path = os.path.join(parent_dir, sub_dir)
        if os.path.isdir(sub_dir_path):
            count = count_images(sub_dir_path)
            print(f"In directory '{sub_dir}', there are {count} image items.")
            total_count += count
    print(f"\nTotal number of Images in the '{parent_dir}' directory folder are {total_count} image items")


In [7]:
def count_images(directory):
    count = 0
    for file in os.listdir(directory):
        if file.endswith(".jpg") or file.endswith(".jpeg") or file.endswith(".png"):
            count += 1
    return count

**This function perform_augmentation takes an image as input and performs random image augmentations on the image. The augmentations that the function performs include `flipping` the image horizontally with a `50% chance`, adjusting the brightness and contrast of the image randomly, applying a blur filter with a `25% chance`, and rotating the image between `-10 and 10 degrees`.
The function uses the Pillow library, which is a fork of the Python Imaging Library (PIL), to perform the image manipulations. The function returns the augmented image.**

In [8]:
def perform_augmentation(image):
    # Randomly flip the image horizontally with a 50% chance
    if random.random() < 0.5:
        image = ImageOps.mirror(image)

    # Randomly adjust brightness and contrast
    enhancer = ImageEnhance.Brightness(image)
    image = enhancer.enhance(random.uniform(0.8, 1.2))

    enhancer = ImageEnhance.Contrast(image)
    image = enhancer.enhance(random.uniform(0.8, 1.2))

    # Randomly apply a blur filter with a 25% chance
    if random.random() < 0.25:
        image = image.filter(ImageFilter.BLUR)

    # Randomly rotate the image between -10 and 10 degrees
    angle = random.uniform(-10, 10)
    image = image.rotate(angle)

    # Return the augmented image
    return image


**This function takes a parent folder path, loops over its subfolders, and performs random image augmentation on images in the subfolders if there are less than 625 images. If there are less than 625 images, the function makes copies of some of the images to bring the total count up to 625. Then it performs image augmentation on all images in the subfolder and saves the augmented images back to the subfolder.**

Here's what the function does step by step:

- Loop over subfolders in the parent folder.
- Check if each subfolder is a directory.
- List all the image files (with a .jpg extension) in the subfolder.
- If there are less than 750 images in the subfolder, create new images by copying some of the existing ones. The number of new images created is 625 minus the number of images in the subfolder.
- Save the new images in a temporary folder.
- For each image file in the temporary folder, perform image augmentation and save the augmented image back to the subfolder with the same filename.
- Delete the temporary folder.

In [9]:
def augment_images(parent_folder):
    for subfolder_path in Path(parent_folder).iterdir():
        if subfolder_path.is_dir():
            images = list(subfolder_path.glob("*.jpg"))
            num_images = len(images)
            if num_images <= 750:
                num_to_copy = 750 - num_images
                temp_folder = Path(parent_folder) / "temp"
                temp_folder.mkdir(exist_ok=True)
                for i in range(num_to_copy):
                    image_to_copy = images[i % num_images]
                    new_filename = f"{subfolder_path.name}_{uuid.uuid4()}.jpg"
                    new_image_path = temp_folder / new_filename
                    shutil.copy(image_to_copy, new_image_path)
                for image_path in temp_folder.glob("*.jpg"):
                    augmented_image = perform_augmentation(Image.open(image_path))
                    new_image_path = subfolder_path / image_path.name
                    augmented_image.save(new_image_path)
                shutil.rmtree(temp_folder)


In [11]:
augment_images("Augmented Image Folder")

In [12]:
count_images_in_subdirs('Augmented Image Folder')

In directory 'AK', there are 750 image items.
In directory 'BCC', there are 750 image items.
In directory 'BKL', there are 750 image items.
In directory 'DF', there are 750 image items.
In directory 'MEL', there are 750 image items.
In directory 'NV', there are 750 image items.
In directory 'SCC', there are 750 image items.
In directory 'VASC', there are 750 image items.

Total number of Images in the 'Augmented Image Folder' directory folder are 6000 image items


**The rename_subfolder_images function takes a parent directory path as input and renames all the image files inside its subdirectories. The new name of each image is a concatenation of its original subdirectory name, a hyphen separator, a UUID, and the original file extension.
Here's the breakdown of what the function does:**

- Loop through all the subdirectories inside the parent directory
- For each subdirectory, loop through all the files inside it
- If the file has an image extension (.jpg, .jpeg, .png), rename it with a new name that includes the subdirectory name and a UUID
- The renamed file has the same extension as the original file

In [13]:
def rename_subfolder_images(parent_dir):
    for sub_dir in os.listdir(parent_dir):
        sub_dir_path = os.path.join(parent_dir, sub_dir)
        if os.path.isdir(sub_dir_path):
            for file in os.listdir(sub_dir_path):
                if file.endswith(".jpg") or file.endswith(".jpeg") or file.endswith(".png"):
                    file_path = os.path.join(sub_dir_path, file)
                    new_file_name = sub_dir + "-" + str(uuid.uuid1()) + os.path.splitext(file)[1]
                    new_file_path = os.path.join(sub_dir_path, new_file_name)
                    os.rename(file_path, new_file_path)

In [14]:
rename_subfolder_images("Augmented Image Folder")

In [15]:
count_images_in_subdirs('Augmented Image Folder')

In directory 'AK', there are 750 image items.
In directory 'BCC', there are 750 image items.
In directory 'BKL', there are 750 image items.
In directory 'DF', there are 750 image items.
In directory 'MEL', there are 750 image items.
In directory 'NV', there are 750 image items.
In directory 'SCC', there are 750 image items.
In directory 'VASC', there are 750 image items.

Total number of Images in the 'Augmented Image Folder' directory folder are 6000 image items
